# DAML HW4 Exploration Studio
## Patterns, representations, and questions to investigate

**Optional and ungraded · 20–45 minutes**

This notebook uses images, toy observations, and pre-specified displays to develop intuition. It leaves the graded convolution, pooling, projection, normalization, masking, attention, and bonus algorithms for you to implement in **DAML_HW04_Searchlight.ipynb**.

For every experiment: **predict → run → explain → change one thing**. Do not submit this notebook. The conceptual questions are prompts for your own reasoning, not model answers to the interpretation response.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(4)
np.set_printoptions(precision=3, suppress=True)

## 1. Which visual differences matter?

Compare these small patterns. Describe a measurement that could distinguish two of them, and one that might fail. Do not implement a detector yet.

Move a stripe or add a small noisy patch. Decide whether its identity should change. Separate what you observe in the pixels from what you would want a model to predict.

In [ ]:
vertical = np.zeros((12, 12)); vertical[2:10, 4:6] = 1.
horizontal = np.zeros((12, 12)); horizontal[5:7, 2:10] = 1.
corner = np.zeros((12, 12)); corner[2:10, 3:5] = 1.; corner[8:10, 3:10] = 1.
patterns = {"vertical stripe": vertical, "horizontal stripe": horizontal, "corner": corner}
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, (name, pixels) in zip(axes, patterns.items()):
    ax.imshow(pixels, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name); ax.axis("off")
plt.show()

## 2. Translation as an experimental variable

The cell below changes the location of a pattern without applying a feature detector. The stripe is kept away from the image boundary so the shift does not wrap it around.

Predict what a useful detector ought to do when the input moves. After you have implemented your homework functions, you can test your prediction independently. For now, record the behavior you expect and how you would measure it.

In [ ]:
moved = np.roll(vertical, shift=2, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, pixels, title in zip(axes, [vertical, moved], ["original", "shifted right"]):
    ax.imshow(pixels, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title); ax.axis("off")
plt.show()

## 3. Design a test of image summarization

Inspect the four regions below. Choose two that you think a summary method could confuse. Write down your proposed summary and a prediction before calculating anything.

Design a second pair that would challenge a different summary. This is a test-design exercise: no pooling result or explanation of the graded interpretation question is supplied.

In [ ]:
regions = [np.array([[0., 0.], [0., 8.]]),
           np.array([[1., 3.], [2., 6.]]),
           np.array([[5., 0.], [0., 1.]]),
           np.array([[2., 4.], [3., 1.]])]
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for i, (ax, region) in enumerate(zip(axes, regions), start=1):
    ax.imshow(region, cmap="viridis", vmin=0, vmax=8)
    for row in range(2):
        for col in range(2):
            ax.text(col, row, str(int(region[row, col])), ha="center", va="center", color="white")
    ax.set_title(f"region {i}"); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 4. Representation choices change what you notice

These coordinates are fictional measurements, not outputs from a learned projection. Compare an observation's brightness with its vertical and horizontal evidence.

Choose two observations that look similar in one pair of coordinates and less similar in another. What extra information would help you choose a representation for a rescue task?

In [ ]:
labels = ["A", "B", "C", "D", "E"]
measurements = np.array([[.9, .1, .3], [.1, .8, .4], [.7, .3, .8],
                         [.2, .7, .9], [.5, .5, .5]])
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, columns, names in [(axes[0], (0, 1), ("vertical evidence", "horizontal evidence")),
                            (axes[1], (0, 2), ("vertical evidence", "brightness"))]:
    ax.scatter(measurements[:, columns[0]], measurements[:, columns[1]])
    for label, point in zip(labels, measurements):
        ax.annotate(label, (point[columns[0]], point[columns[1]]))
    ax.set_xlabel(names[0]); ax.set_ylabel(names[1]); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 5. Read an attention display without computing attention

The following weights are **invented examples**, entered directly for visualization. They were not calculated by an attention implementation.

For each row, identify which observation receives the largest weight. Propose a possible task associated with that pattern, and explain what the display cannot tell you about how the weights were obtained. Do not treat the numbers as homework expected outputs.

In [ ]:
illustrative_weights = np.array([[.55, .10, .20, .05, .10],
                                 [.10, .40, .05, .30, .15],
                                 [.15, .15, .20, .20, .30]])
fig, ax = plt.subplots(figsize=(7, 3))
plot = ax.imshow(illustrative_weights, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(5), labels); ax.set_yticks(range(3), ["scenario 1", "scenario 2", "scenario 3"])
ax.set_xlabel("observation"); ax.set_title("Invented weight patterns")
fig.colorbar(plot, ax=ax, label="weight")
plt.show()

## 6. Concentration: describe first, measure later

Compare the two hand-specified distributions. Describe their differences in words. Invent an intermediate distribution and plot it.

What properties would you want a numerical concentration measure to have? Write your own criteria. The entropy formula and implementation are deliberately left to the optional homework task.

In [ ]:
distributions = {"pattern A": np.array([.7, .1, .1, .1]),
                 "pattern B": np.array([.25, .25, .25, .25])}
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, (name, values) in zip(axes, distributions.items()):
    ax.bar(range(4), values); ax.set_ylim(0, 1); ax.set_title(name)
    ax.set_xlabel("observation"); ax.set_ylabel("weight")
plt.tight_layout(); plt.show()

## 7. Design a sequential experiment

Imagine a drone receives observations at times 1 through 6 and makes a decision after each arrival. Draw what information you think should be available at each decision time. Explain your reasoning before consulting your lecture notes.

Propose a test that would distinguish a system following that information rule from one that violates it. The mask construction and interpretation answer belong in your homework; this notebook does not supply them.

## 8. Multiple views of the same scene

Sketch two different questions a system might ask about the same scene. Predict whether the answers would focus on the same observations. What evidence would convince you that two views provide complementary information?

After completing the homework, turn one of your predictions into an experiment with your own implementation. The multi-head algorithm is not supplied here.

## Choose one extension

- Change one visual pattern while keeping its label meaningful; define a robustness test.
- Design a pair of inputs that exposes a limitation of your chosen summary.
- Compare two hand-drawn weight displays and explain their differences.
- Keep an experiment log: hypothesis, controlled change, observation, unresolved question.

Use your course lectures for the underlying theory. Start with your own predictions and implementation before consulting external worked-code tutorials.